# 상위 10칸에 남은 구매금액 진단 (학습 없음)
학습된 모형 20개(M1·M2·M4·M5 × 시드 42~46)의 **저장된 체크포인트를 다시 채점**만 합니다. 새로 학습하는 것은 없고 몇 분이면 끝납니다.

**보는 것 1 — 여지가 있는가.** 고객이 평가 기간에 실제로 산 상품의 구매금액을 세 갈래로 나눕니다: 이미 상위 10에 든 금액 / 11~50위에 있어 재정렬로 끌어올 수 있는 금액 / 50위 밖이라 손댈 수 없는 금액. 여기에 상한선(상위 50 안에서 금액이 가장 큰 10개를 고른 경우)을 함께 냅니다. 상한선은 정답을 보고 고른 값이라 **방법이 아니라 천장**입니다.

**보는 것 2 — 실제 규칙으로 얼마나 잡히는가.** 같은 후보 50개를 `(1-w)×순위가중 + w×기대 구매금액`으로 다시 정렬합니다. `w = alpha × q_C(고객의 CLV 백분위)`라서 고CLV 고객에게만, 그 고객의 CLV만큼만 가치 쪽으로 밉니다. 기대 구매금액은 학습 구간에서만 계산합니다. alpha를 0/0.1/0.2/0.4로 훑어 정확도와 가치의 교환 곡선을 그립니다.

alpha는 **고르지 않습니다.** 곡선만 보고합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'ceb0b37f7e0ea51fe022c69697a1e4d0eab36257'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
import torch
import lightgcn_clv_value_rerank_diagnostic as diag

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = diag.configure_rerank_diagnostic()
summary = diag.preflight_summary(cfg)
assert summary['trains_anything'] is False
assert summary['alphas'][0] == 0.0
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
headroom = diag.run_value_rerank_diagnostic(cfg)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) 상위 10 밖에 남은 구매금액 (고객 1인 평균, 5시드 평균)')
show(headroom)
print('2) CLV 게이트 가치 재정렬 스윕 (alpha=0이 재정렬 없음)')
show(headroom.attrs['rerank'])
print('3) 판독')
print(json.dumps(headroom.attrs['reading'], ensure_ascii=False, indent=2))
print('저장 파일:', json.dumps(headroom.attrs['result_paths'], ensure_ascii=False, indent=2))
